In [2]:
# -*- coding: utf-8 -*-
r"""
FINAL CODE (CSV version) — State-wise sheets in TWO Excel files

Outputs (same logic as before, but each workbook contains one sheet per State):
  D:\Dailydata_tenderdetails\Final_Data\Live_Tender.xlsx
  D:\Dailydata_tenderdetails\Final_Data\Past_Tender.xlsx

Logic (unchanged):
  1) LIVE:
     - Merge CSVs from Di_live, Ductile_Iron_live, Water_live -> df_keyword (dedup by tender_id)
     - Merge CSVs from Mail_Live -> df_mail (dedup by tender_id)
     - Remove from df_mail any tender_id already in df_keyword
     - df_merge_live = df_keyword ∪ remaining df_mail (dedup by tender_id)
  2) PAST:
     - Merge CSVs from Di_pipe_close, Ductile_iron_close, Water_close -> df_merge_past (dedup by tender_id)
  3) FINAL FILTER:
     - Remove from df_merge_live any tender_id present in df_merge_past
  4) CLEANUPS:
     - tender_id uppercase (and strip trailing ".0")
     - if tender_id is purely numeric and starts with '0' → remove leading zeros
     - add "Cr." column right after "Tender Amount" (convert to Crore; original amount unchanged)
  5) SHAPE to final schema (missing source columns become blank)
     - Dates normalized to DD-MM-YYYY for: Entry Date, Publish Date, Bid Submission Date
  6) SAVE as TWO separate Excel files, each with multiple **state-wise** sheets

Tip: If a State cell is empty or missing, its rows go to the sheet "Unknown".
"""

import os
import glob
import re
from typing import Optional, List, Dict
import pandas as pd
from datetime import datetime

# ========= PATHS =========
BASE_DIR_INPUT = r"D:\Dailydata_tenderdetails"                 # input root (unchanged)
OUTPUT_DIR     = r"D:\Dailydata_tenderdetails\Final_Data"      # output folder (unchanged)

# LIVE folders
LIVE_DI       = os.path.join(BASE_DIR_INPUT, "live_data", "Di_live")
LIVE_DI_IRON  = os.path.join(BASE_DIR_INPUT, "live_data", "Ductile_Iron_live")
LIVE_WATER    = os.path.join(BASE_DIR_INPUT, "live_data", "Water_live")
LIVE_MAIL     = os.path.join(BASE_DIR_INPUT, "live_data", "Mail_Live")

# PAST folders
PAST_DI_PIPE  = os.path.join(BASE_DIR_INPUT, "Close_data", "Di_pipe_close")
PAST_DI_IRON  = os.path.join(BASE_DIR_INPUT, "Close_data", "Ductile_iron_close")
PAST_WATER    = os.path.join(BASE_DIR_INPUT, "Close_data", "Water_close")

# Output files (two workbooks)
OUT_LIVE_XLSX = os.path.join(OUTPUT_DIR, "Live_Tender.xlsx")
OUT_PAST_XLSX = os.path.join(OUTPUT_DIR, "Past_Tender.xlsx")

# ========= CONFIG / CONSTANTS =========
CSV_PATTERNS: tuple = ("*.csv", "*.CSV")
AMOUNT_PRIMARY_NAME = "Tender Amount"  # where we try to insert "Cr." after
DATE_FMT = "%d-%m-%Y"  # DD-MM-YYYY

# Columns mapping helpers
TENDER_ID_ALIASES = {
    "tender_id","tender id","tenderid",
    "tender no","tender number","tender#",
    "tender ref no","tender ref no.","tender reference no",
    "tender reference number","tender reference id",
    "notice no","notice number",
    "tdr","tdr id","tdr_no","tdr no","tdr number",
    "ref no","reference no","reference number",
}

FINAL_COLS: List[str] = [
    "Entry Date",
    "Project Scheme",
    "State",
    "Location",
    "Tender Site",
    "Project Name",
    "Project Link",
    "TDR No",
    "Department Name",
    "Tender ID",
    "Refrence No",
    "Tender Value(Crore)",
    "Tender Status",
    "Publish Date",
    "Bid Submission Date",
    "Department Address",
    "Contractor Name(L1)",
    "Bidder Name",
]

_num_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

# ========= SMALL, FOCUSED HELPERS =========
def list_csv_files(folder: str) -> List[str]:
    """Recursively list CSV files in a folder."""
    if not os.path.isdir(folder):
        return []
    files: List[str] = []
    for pat in CSV_PATTERNS:
        files += glob.glob(os.path.join(folder, "**", pat), recursive=True)
    return sorted(files)

def read_csv_safely(path: str) -> pd.DataFrame:
    """Read CSV with common encodings; return empty DF on failure."""
    for enc in ("utf-8-sig", "utf-8", "latin1"):
        try:
            df = pd.read_csv(path, dtype=str, encoding=enc)
            return df if not df.empty else pd.DataFrame()
        except Exception:
            continue
    return pd.DataFrame()

def clean_tender_id_value(x: str) -> str:
    """
    tender_id rules:
      - trim; strip trailing '.0'
      - if purely digits: remove leading zeros (keep '0' if all zeros)
      - else: uppercase
    """
    s = "" if pd.isna(x) else str(x).strip()
    s = re.sub(r"\.0$", "", s)
    if s.isdigit():
        s = s.lstrip("0")
        return s if s != "" else "0"
    return s.upper()

def normalize_tender_id(df: pd.DataFrame) -> pd.DataFrame:
    """Rename ID aliases to 'tender_id' and apply cleaning rules."""
    if df.empty:
        return df
    df = df.rename(columns={c: c.strip() for c in df.columns})
    fold = {c.casefold().strip(): c for c in df.columns}
    found = None
    for alias in TENDER_ID_ALIASES:
        if alias.casefold() in fold:
            found = fold[alias.casefold()]
            break
    if found and found != "tender_id":
        df = df.rename(columns={found: "tender_id"})
    if "tender_id" not in df.columns:
        df["tender_id"] = pd.NA
    df["tender_id"] = df["tender_id"].apply(clean_tender_id_value)
    return df.dropna(how="all")

def merge_folders_to_df(*folders: str) -> pd.DataFrame:
    """Read & stack all CSV files from the given folders (with tender_id normalization)."""
    frames: List[pd.DataFrame] = []
    for folder in folders:
        for f in list_csv_files(folder):
            df = read_csv_safely(f)
            if not df.empty:
                frames.append(normalize_tender_id(df))
    if not frames:
        return pd.DataFrame(columns=["tender_id"])
    return pd.concat(frames, ignore_index=True, sort=False)

def dedupe_by_tender_id(df: pd.DataFrame) -> pd.DataFrame:
    """Deduplicate by tender_id (blank IDs dedup by all columns)."""
    if df.empty:
        return df
    if "tender_id" in df.columns:
        df["tender_id"] = df["tender_id"].astype(str).str.strip()
        non_blank = df[df["tender_id"] != ""].drop_duplicates(subset=["tender_id"], keep="first")
        blank     = df[df["tender_id"] == ""].drop_duplicates(keep="first")
        return pd.concat([non_blank, blank], ignore_index=True, sort=False)
    return df.drop_duplicates(keep="first")

def filter_out_ids(df: pd.DataFrame, ids: set) -> pd.DataFrame:
    """Remove rows whose tender_id belongs to 'ids'."""
    if df.empty or not ids or "tender_id" not in df.columns:
        return df
    return df[~df["tender_id"].astype(str).isin(ids)].copy()

def parse_amount_to_crore(val: str) -> float:
    """
    Convert text amount to crores.
      - '1.5 Cr' or '1.5 Crore' -> 1.5
      - '75 Lakh' / '75 Lac'    -> 0.75
      - '₹12,34,56,789' (rupees)-> 12.3456789
      - else NaN
    """
    if pd.isna(val):
        return float("nan")
    s = str(val).strip()
    if not s:
        return float("nan")
    low = s.lower()

    if "crore" in low or re.search(r"\bcr\b", low):
        m = _num_re.search(low)
        return float(m.group(0)) if m else float("nan")

    if "lakh" in low or "lac" in low:
        m = _num_re.search(low)
        return (float(m.group(0)) / 100.0) if m else float("nan")

    cleaned = re.sub(r"[^\d.]", "", s)
    if cleaned in ("", "."):
        return float("nan")
    try:
        rupees = float(cleaned)
    except Exception:
        return float("nan")
    return rupees / 1e7

def insert_crore_column(df: pd.DataFrame) -> pd.DataFrame:
    """Insert 'Cr.' column immediately after an amount column (Tender Amount or similar)."""
    if df.empty:
        return df

    # Prefer exact "Tender Amount"
    target_col = next((c for c in df.columns if c.strip().lower() == AMOUNT_PRIMARY_NAME.lower()), None)
    # Fallbacks containing words 'tender' and 'amount'
    if target_col is None:
        for c in df.columns:
            lc = c.strip().lower()
            if "tender" in lc and "amount" in lc:
                target_col = c
                break
    # More fallbacks
    if target_col is None:
        for alt in ("tenderamount", "estimated cost", "estimated value", "value", "tender amount (rs)"):
            for c in df.columns:
                if c.strip().lower() == alt:
                    target_col = c
                    break
            if target_col:
                break

    if target_col is None:
        return df  # no amount column found

    crore_series = df[target_col].apply(parse_amount_to_crore)

    insert_at = list(df.columns).index(target_col) + 1
    if "Cr." in df.columns:
        df = df.drop(columns=["Cr."])
    left_cols  = list(df.columns)[:insert_at]
    right_cols = list(df.columns)[insert_at:]
    df_left = df[left_cols].copy()
    df_right = df[right_cols].copy()
    df_left["Cr."] = crore_series
    return pd.concat([df_left, df_right], axis=1)

def find_col(df: pd.DataFrame, aliases: List[str]) -> Optional[str]:
    """Return actual column name matching any alias (case-insensitive, trimmed)."""
    if df.empty:
        return None
    amap = {c.strip().lower(): c for c in df.columns}
    for a in aliases:
        key = a.strip().lower()
        if key in amap:
            return amap[key]
    # allow 'contains all words' matching
    for a in aliases:
        parts = [p for p in a.strip().lower().split() if p]
        for c in df.columns:
            lc = c.strip().lower()
            if all(p in lc for p in parts):
                return c
    return None

# ======== Date normalization helpers (DD-MM-YYYY) ========
def _normalize_date_value(v: str) -> str:
    """Return DD-MM-YYYY if parsed, else original/stringified value."""
    if pd.isna(v):
        return ""
    s = str(v).strip()
    if not s:
        return ""

    # Handle Excel-like numeric serials (e.g., "45213" or "45213.0")
    try:
        num = float(s)
        if 20000 <= int(round(num)) <= 90000:  # plausible range
            dt = pd.to_datetime(int(round(num)), origin="1899-12-30", unit="D", errors="coerce")
            if not pd.isna(dt):
                return dt.strftime(DATE_FMT)
    except Exception:
        pass

    # Try parsing with dayfirst=True, then fallback (no infer_datetime_format)
    dt = pd.to_datetime(s, dayfirst=True, errors="coerce")
    if pd.isna(dt):
        dt = pd.to_datetime(s, dayfirst=False, errors="coerce")
    return dt.strftime(DATE_FMT) if not pd.isna(dt) else s

def _normalize_date_series(series: Optional[pd.Series]) -> pd.Series:
    """Vectorized DD-MM-YYYY normalization, blanks if None."""
    if series is None:
        return pd.Series([], dtype=str)
    return series.apply(_normalize_date_value)

def to_final_schema(df: pd.DataFrame) -> pd.DataFrame:
    """
    Map raw df to the requested final column structure.
    If a source column is missing, output column stays blank.
    Dates are normalized to DD-MM-YYYY.
    """
    if df.empty:
        return pd.DataFrame(columns=FINAL_COLS)

    # Entry Date in DD-MM-YYYY
    today_str = datetime.now().strftime(DATE_FMT)

    col_state      = find_col(df, ["State"])
    col_location   = find_col(df, ["Location", "City"])
    col_tdr        = find_col(df, ["TDR", "TDR No", "TDRNo", "TDR Number"])
    col_auth       = find_col(df, ["Tendering Authority", "Department Name", "Authority"])
    col_tid        = find_col(df, ["tender_id", "Tender ID"])
    col_tender_no  = find_col(df, ["TenderNo", "Tender No", "Tender Number", "Reference No", "Ref No"])
    col_brief      = find_col(df, ["Tender Brief", "Project Name", "Title"])
    col_pubdate    = find_col(df, ["pubDate", "Publish Date", "PublishDate"])
    col_duedate    = find_col(df, ["DueDate", "Due Date", "Bid Submission Date"])
    col_crore      = find_col(df, ["Cr.", "Tender Value(Crore)"])

    # If "Cr." not present (rare), compute from any amount column
    if col_crore is None:
        col_amt = find_col(df, [AMOUNT_PRIMARY_NAME, "TenderAmount", "Amount", "Tender Amount (Rs)", "Estimated Cost", "Estimated Value", "Value"])
        if col_amt:
            crore_series = df[col_amt].apply(parse_amount_to_crore)
        else:
            crore_series = pd.Series([float("nan")] * len(df))
    else:
        crore_series = pd.to_numeric(df[col_crore], errors="coerce")

    # Normalize date columns to DD-MM-YYYY
    pub_series = _normalize_date_series(df[col_pubdate]) if col_pubdate else pd.Series([""] * len(df))
    due_series = _normalize_date_series(df[col_duedate]) if col_duedate else pd.Series([""] * len(df))

    out = pd.DataFrame({
        "Entry Date":              today_str,
        "Project Scheme":          "",
        "State":                   df[col_state]    if col_state    else "",
        "Location":                df[col_location] if col_location else "",
        "Tender Site":             "Tender Details",
        "Project Name":            df[col_brief]    if col_brief    else "",
        "Project Link":            "",
        "TDR No":                  df[col_tdr]      if col_tdr      else "",
        "Department Name":         df[col_auth]     if col_auth     else "",
        "Tender ID":               df[col_tid]      if col_tid      else "",
        "Refrence No":             df[col_tender_no]if col_tender_no else "",
        "Tender Value(Crore)":     crore_series.round(2),
        "Tender Status":           "",
        "Publish Date":            pub_series,
        "Bid Submission Date":     due_series,
        "Department Address":      "",
        "Contractor Name(L1)":     "",
        "Bidder Name":             "",
    }, columns=FINAL_COLS)

    # Stringify non-numeric columns, replace NaN with blanks
    for col in out.columns:
        if col != "Tender Value(Crore)":
            out[col] = out[col].astype(str).replace({"nan": ""})
    return out

# ========= STATE-WISE WRITER =========
def _sanitize_sheet_name(name: str) -> str:
    """Make a safe Excel sheet name (<=31 chars, no []:*?/\\)."""
    if pd.isna(name) or str(name).strip() == "":
        name = "Unknown"
    invalid = r'[]:*?/\\'
    for ch in invalid:
        name = name.replace(ch, "_")
    name = name.strip()
    return name[:31] if len(name) > 31 else name

def write_statewise_workbook(path: str, df: pd.DataFrame) -> None:
    """
    Write a workbook with one sheet per State.
    - If 'State' column missing, write one sheet 'All'.
    - If some states are blank, they go to 'Unknown'.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)

    if df.empty:
        with pd.ExcelWriter(path, engine="openpyxl") as writer:
            empty_df = pd.DataFrame(columns=FINAL_COLS)
            empty_df.to_excel(writer, sheet_name="Empty", index=False)
        return

    if "State" not in df.columns:
        with pd.ExcelWriter(path, engine="openpyxl") as writer:
            df.to_excel(writer, sheet_name="All", index=False)
        return

    used: Dict[str, int] = {}
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        states_series = df["State"].fillna("").astype(str)
        unique_states = sorted(set(states_series))
        if "" in unique_states:
            unique_states.remove("")
            unique_states = [""] + unique_states  # 'Unknown' first

        for state_value in unique_states:
            sheet_df = df[states_series == state_value].copy()
            safe = _sanitize_sheet_name(state_value)
            if safe in used:
                used[safe] += 1
                safe = f"{safe[:28]}_{used[safe]:02d}" if len(safe) > 28 else f"{safe}_{used[safe]:02d}"
            else:
                used[safe] = 1
            sheet_df.to_excel(writer, sheet_name=safe, index=False)

# ========= MAIN PIPELINE =========
if __name__ == "__main__":
    print("========== START ==========")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ---- LIVE: keywords ----
    print("[STEP] LIVE keywords merge")
    df_keyword = dedupe_by_tender_id(merge_folders_to_df(LIVE_DI, LIVE_DI_IRON, LIVE_WATER))
    print(f"[INFO] df_keyword rows: {len(df_keyword)}")

    # ---- LIVE: mail ----
    print("[STEP] LIVE mail merge")
    df_mail = dedupe_by_tender_id(merge_folders_to_df(LIVE_MAIL))
    print(f"[INFO] df_mail rows: {len(df_mail)}")

    # Remove overlaps (mail - keyword)
    ids_keyword = set(df_keyword["tender_id"].dropna().astype(str)) if not df_keyword.empty else set()
    df_mail_unique = filter_out_ids(df_mail, ids_keyword)
    print(f"[INFO] df_mail_unique rows: {len(df_mail_unique)}")

    # Final LIVE merge & dedupe
    df_merge_live = dedupe_by_tender_id(
        pd.concat([df_keyword, df_mail_unique], ignore_index=True, sort=False)
        if not (df_keyword.empty and df_mail_unique.empty) else pd.DataFrame(columns=["tender_id"])
    )
    print(f"[INFO] df_merge_live rows (pre-past-filter): {len(df_merge_live)}")

    # ---- PAST ----
    print("[STEP] PAST merge")
    df_merge_past = dedupe_by_tender_id(merge_folders_to_df(PAST_DI_PIPE, PAST_DI_IRON, PAST_WATER))
    print(f"[INFO] df_merge_past rows: {len(df_merge_past)}")

    # ---- FINAL FILTER: LIVE minus PAST by tender_id ----
    ids_past = set(df_merge_past["tender_id"].dropna().astype(str)) if not df_merge_past.empty else set()
    df_merge_live = dedupe_by_tender_id(filter_out_ids(df_merge_live, ids_past))
    print(f"[INFO] df_merge_live rows (final live before shaping): {len(df_merge_live)}")

    # ---- Add/refresh "Cr." column on BOTH frames ----
    df_merge_live = insert_crore_column(df_merge_live)
    df_merge_past = insert_crore_column(df_merge_past)

    # ---- SHAPE to final schema (dates normalized to DD-MM-YYYY) ----
    live_final = to_final_schema(df_merge_live)
    past_final = to_final_schema(df_merge_past)

    # ---- SAVE: one workbook per output, with multiple sheets (state-wise) ----
    print("[STEP] Writing state-wise workbooks")
    write_statewise_workbook(OUT_LIVE_XLSX, live_final)
    write_statewise_workbook(OUT_PAST_XLSX, past_final)

    print(f"[OK] Live_Tender  -> {OUT_LIVE_XLSX}")
    print(f"[OK] Past_Tender  -> {OUT_PAST_XLSX}")
    print("=========== DONE ==========")


========== START ==========
[STEP] LIVE keywords merge
[INFO] df_keyword rows: 4243
[STEP] LIVE mail merge
[INFO] df_mail rows: 4694
[INFO] df_mail_unique rows: 4694
[INFO] df_merge_live rows (pre-past-filter): 8937
[STEP] PAST merge
[INFO] df_merge_past rows: 2245
[INFO] df_merge_live rows (final live before shaping): 6550
[STEP] Writing state-wise workbooks
[OK] Live_Tender  -> D:\Dailydata_tenderdetails\Final_Data\Live_Tender.xlsx
[OK] Past_Tender  -> D:\Dailydata_tenderdetails\Final_Data\Past_Tender.xlsx
=========== DONE ==========


In [1]:
# -*- coding: utf-8 -*-
r"""
FINAL CODE (CSV version) — State-wise sheets in TWO Excel files

Outputs (same logic as before, but each workbook contains one sheet per State + an 'All Data' sheet first):
  D:\Dailydata_tenderdetails\Final_Data\Live_Tender.xlsx
  D:\Dailydata_tenderdetails\Final_Data\Past_Tender.xlsx

Logic (unchanged):
  1) LIVE:
     - Merge CSVs from Di_live, Ductile_Iron_live, Water_live -> df_keyword (dedup by tender_id)
     - Merge CSVs from Mail_Live -> df_mail (dedup by tender_id)
     - Remove from df_mail any tender_id already in df_keyword
     - df_merge_live = df_keyword ∪ remaining df_mail (dedup by tender_id)
  2) PAST:
     - Merge CSVs from Di_pipe_close, Ductile_iron_close, Water_close -> df_merge_past (dedup by tender_id)
  3) FINAL FILTER:
     - Remove from df_merge_live any tender_id present in df_merge_past
  4) CLEANUPS:
     - tender_id uppercase (and strip trailing ".0")
     - if tender_id is purely numeric and starts with '0' → remove leading zeros
     - add "Cr." column right after "Tender Amount" (convert to Crore; original amount unchanged)
  5) SHAPE to final schema (missing source columns become blank)
     - Dates normalized to DD-MM-YYYY for: Entry Date, Publish Date, Bid Submission Date
  6) SAVE as TWO separate Excel files:
     - First sheet: "All Data" (entire dataset)
     - Next sheets: one per State (blank/NA -> "Unknown")

Tip: If a State cell is empty or missing, those rows go to the sheet "Unknown".
"""

import os
import glob
import re
from typing import Optional, List, Dict
import pandas as pd
from datetime import datetime

# ========= PATHS =========
BASE_DIR_INPUT = r"D:\Dailydata_tenderdetails"                 # input root (unchanged)
OUTPUT_DIR     = r"D:\Dailydata_tenderdetails\Final_Data"      # output folder (unchanged)

# LIVE folders
LIVE_DI       = os.path.join(BASE_DIR_INPUT, "live_data", "Di_live")
LIVE_DI_IRON  = os.path.join(BASE_DIR_INPUT, "live_data", "Ductile_Iron_live")
LIVE_WATER    = os.path.join(BASE_DIR_INPUT, "live_data", "Water_live")
LIVE_MAIL     = os.path.join(BASE_DIR_INPUT, "live_data", "Mail_Live")

# PAST folders
PAST_DI_PIPE  = os.path.join(BASE_DIR_INPUT, "Close_data", "Di_pipe_close")
PAST_DI_IRON  = os.path.join(BASE_DIR_INPUT, "Close_data", "Ductile_iron_close")
PAST_WATER    = os.path.join(BASE_DIR_INPUT, "Close_data", "Water_close")

# Output files (two workbooks)
OUT_LIVE_XLSX = os.path.join(OUTPUT_DIR, "Live_Tender.xlsx")
OUT_PAST_XLSX = os.path.join(OUTPUT_DIR, "Past_Tender.xlsx")

# ========= CONFIG / CONSTANTS =========
CSV_PATTERNS: tuple = ("*.csv", "*.CSV")
AMOUNT_PRIMARY_NAME = "Tender Amount"  # where we try to insert "Cr." after
DATE_FMT = "%d-%m-%Y"  # DD-MM-YYYY

# Columns mapping helpers
TENDER_ID_ALIASES = {
    "tender_id","tender id","tenderid",
    "tender no","tender number","tender#",
    "tender ref no","tender ref no.","tender reference no",
    "tender reference number","tender reference id",
    "notice no","notice number",
    "tdr","tdr id","tdr_no","tdr no","tdr number",
    "ref no","reference no","reference number",
}

FINAL_COLS: List[str] = [
    "Entry Date",
    "Project Scheme",
    "State",
    "Location",
    "Tender Site",
    "Project Name",
    "Project Link",
    "TDR No",
    "Department Name",
    "Tender ID",
    "Refrence No",
    "Tender Value(Crore)",
    "Tender Status",
    "Publish Date",
    "Bid Submission Date",
    "Department Address",
    "Contractor Name(L1)",
    "Bidder Name",
]

_num_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

# ========= SMALL, FOCUSED HELPERS =========
def list_csv_files(folder: str) -> List[str]:
    """Recursively list CSV files in a folder."""
    if not os.path.isdir(folder):
        return []
    files: List[str] = []
    for pat in CSV_PATTERNS:
        files += glob.glob(os.path.join(folder, "**", pat), recursive=True)
    return sorted(files)

def read_csv_safely(path: str) -> pd.DataFrame:
    """Read CSV with common encodings; return empty DF on failure."""
    for enc in ("utf-8-sig", "utf-8", "latin1"):
        try:
            df = pd.read_csv(path, dtype=str, encoding=enc)
            return df if not df.empty else pd.DataFrame()
        except Exception:
            continue
    return pd.DataFrame()

def clean_tender_id_value(x: str) -> str:
    """
    tender_id rules:
      - trim; strip trailing '.0'
      - if purely digits: remove leading zeros (keep '0' if all zeros)
      - else: uppercase
    """
    s = "" if pd.isna(x) else str(x).strip()
    s = re.sub(r"\.0$", "", s)
    if s.isdigit():
        s = s.lstrip("0")
        return s if s != "" else "0"
    return s.upper()

def normalize_tender_id(df: pd.DataFrame) -> pd.DataFrame:
    """Rename ID aliases to 'tender_id' and apply cleaning rules."""
    if df.empty:
        return df
    df = df.rename(columns={c: c.strip() for c in df.columns})
    fold = {c.casefold().strip(): c for c in df.columns}
    found = None
    for alias in TENDER_ID_ALIASES:
        if alias.casefold() in fold:
            found = fold[alias.casefold()]
            break
    if found and found != "tender_id":
        df = df.rename(columns={found: "tender_id"})
    if "tender_id" not in df.columns:
        df["tender_id"] = pd.NA
    df["tender_id"] = df["tender_id"].apply(clean_tender_id_value)
    return df.dropna(how="all")

def merge_folders_to_df(*folders: str) -> pd.DataFrame:
    """Read & stack all CSV files from the given folders (with tender_id normalization)."""
    frames: List[pd.DataFrame] = []
    for folder in folders:
        for f in list_csv_files(folder):
            df = read_csv_safely(f)
            if not df.empty:
                frames.append(normalize_tender_id(df))
    if not frames:
        return pd.DataFrame(columns=["tender_id"])
    return pd.concat(frames, ignore_index=True, sort=False)

def dedupe_by_tender_id(df: pd.DataFrame) -> pd.DataFrame:
    """Deduplicate by tender_id (blank IDs dedup by all columns)."""
    if df.empty:
        return df
    if "tender_id" in df.columns:
        df["tender_id"] = df["tender_id"].astype(str).str.strip()
        non_blank = df[df["tender_id"] != ""].drop_duplicates(subset=["tender_id"], keep="first")
        blank     = df[df["tender_id"] == ""].drop_duplicates(keep="first")
        return pd.concat([non_blank, blank], ignore_index=True, sort=False)
    return df.drop_duplicates(keep="first")

def filter_out_ids(df: pd.DataFrame, ids: set) -> pd.DataFrame:
    """Remove rows whose tender_id belongs to 'ids'."""
    if df.empty or not ids or "tender_id" not in df.columns:
        return df
    return df[~df["tender_id"].astype(str).isin(ids)].copy()

def parse_amount_to_crore(val: str) -> float:
    """
    Convert text amount to crores.
      - '1.5 Cr' or '1.5 Crore' -> 1.5
      - '75 Lakh' / '75 Lac'    -> 0.75
      - '₹12,34,56,789' (rupees)-> 12.3456789
      - else NaN
    """
    if pd.isna(val):
        return float("nan")
    s = str(val).strip()
    if not s:
        return float("nan")
    low = s.lower()

    if "crore" in low or re.search(r"\bcr\b", low):
        m = _num_re.search(low)
        return float(m.group(0)) if m else float("nan")

    if "lakh" in low or "lac" in low:
        m = _num_re.search(low)
        return (float(m.group(0)) / 100.0) if m else float("nan")

    cleaned = re.sub(r"[^\d.]", "", s)
    if cleaned in ("", "."):
        return float("nan")
    try:
        rupees = float(cleaned)
    except Exception:
        return float("nan")
    return rupees / 1e7

def insert_crore_column(df: pd.DataFrame) -> pd.DataFrame:
    """Insert 'Cr.' column immediately after an amount column (Tender Amount or similar)."""
    if df.empty:
        return df

    # Prefer exact "Tender Amount"
    target_col = next((c for c in df.columns if c.strip().lower() == AMOUNT_PRIMARY_NAME.lower()), None)
    # Fallbacks containing words 'tender' and 'amount'
    if target_col is None:
        for c in df.columns:
            lc = c.strip().lower()
            if "tender" in lc and "amount" in lc:
                target_col = c
                break
    # More fallbacks
    if target_col is None:
        for alt in ("tenderamount", "estimated cost", "estimated value", "value", "tender amount (rs)"):
            for c in df.columns:
                if c.strip().lower() == alt:
                    target_col = c
                    break
            if target_col:
                break

    if target_col is None:
        return df  # no amount column found

    crore_series = df[target_col].apply(parse_amount_to_crore)

    insert_at = list(df.columns).index(target_col) + 1
    if "Cr." in df.columns:
        df = df.drop(columns=["Cr."])
    left_cols  = list(df.columns)[:insert_at]
    right_cols = list(df.columns)[insert_at:]
    df_left = df[left_cols].copy()
    df_right = df[right_cols].copy()
    df_left["Cr."] = crore_series
    return pd.concat([df_left, df_right], axis=1)

def find_col(df: pd.DataFrame, aliases: List[str]) -> Optional[str]:
    """Return actual column name matching any alias (case-insensitive, trimmed)."""
    if df.empty:
        return None
    amap = {c.strip().lower(): c for c in df.columns}
    for a in aliases:
        key = a.strip().lower()
        if key in amap:
            return amap[key]
    # allow 'contains all words' matching
    for a in aliases:
        parts = [p for p in a.strip().lower().split() if p]
        for c in df.columns:
            lc = c.strip().lower()
            if all(p in lc for p in parts):
                return c
    return None

# ======== Date normalization helpers (DD-MM-YYYY) ========
def _normalize_date_value(v: str) -> str:
    """Return DD-MM-YYYY if parsed, else original/stringified value."""
    if pd.isna(v):
        return ""
    s = str(v).strip()
    if not s:
        return ""

    # Handle Excel-like numeric serials (e.g., "45213" or "45213.0")
    try:
        num = float(s)
        if 20000 <= int(round(num)) <= 90000:  # plausible range
            dt = pd.to_datetime(int(round(num)), origin="1899-12-30", unit="D", errors="coerce")
            if not pd.isna(dt):
                return dt.strftime(DATE_FMT)
    except Exception:
        pass

    # Try parsing with dayfirst=True, then fallback (no infer_datetime_format)
    dt = pd.to_datetime(s, dayfirst=True, errors="coerce")
    if pd.isna(dt):
        dt = pd.to_datetime(s, dayfirst=False, errors="coerce")
    return dt.strftime(DATE_FMT) if not pd.isna(dt) else s

def _normalize_date_series(series: Optional[pd.Series]) -> pd.Series:
    """Vectorized DD-MM-YYYY normalization, blanks if None."""
    if series is None:
        return pd.Series([], dtype=str)
    return series.apply(_normalize_date_value)

def to_final_schema(df: pd.DataFrame) -> pd.DataFrame:
    """
    Map raw df to the requested final column structure.
    If a source column is missing, output column stays blank.
    Dates are normalized to DD-MM-YYYY.
    """
    if df.empty:
        return pd.DataFrame(columns=FINAL_COLS)

    # Entry Date in DD-MM-YYYY
    today_str = datetime.now().strftime(DATE_FMT)

    col_state      = find_col(df, ["State"])
    col_location   = find_col(df, ["Location", "City"])
    col_tdr        = find_col(df, ["TDR", "TDR No", "TDRNo", "TDR Number"])
    col_auth       = find_col(df, ["Tendering Authority", "Department Name", "Authority"])
    col_tid        = find_col(df, ["tender_id", "Tender ID"])
    col_tender_no  = find_col(df, ["TenderNo", "Tender No", "Tender Number", "Reference No", "Ref No"])
    col_brief      = find_col(df, ["Tender Brief", "Project Name", "Title"])
    col_pubdate    = find_col(df, ["pubDate", "Publish Date", "PublishDate"])
    col_duedate    = find_col(df, ["DueDate", "Due Date", "Bid Submission Date"])
    col_crore      = find_col(df, ["Cr.", "Tender Value(Crore)"])

    # If "Cr." not present (rare), compute from any amount column
    if col_crore is None:
        col_amt = find_col(df, [AMOUNT_PRIMARY_NAME, "TenderAmount", "Amount", "Tender Amount (Rs)", "Estimated Cost", "Estimated Value", "Value"])
        if col_amt:
            crore_series = df[col_amt].apply(parse_amount_to_crore)
        else:
            crore_series = pd.Series([float("nan")] * len(df))
    else:
        crore_series = pd.to_numeric(df[col_crore], errors="coerce")

    # Normalize date columns to DD-MM-YYYY
    pub_series = _normalize_date_series(df[col_pubdate]) if col_pubdate else pd.Series([""] * len(df))
    due_series = _normalize_date_series(df[col_duedate]) if col_duedate else pd.Series([""] * len(df))

    out = pd.DataFrame({
        "Entry Date":              today_str,
        "Project Scheme":          "",
        "State":                   df[col_state]    if col_state    else "",
        "Location":                df[col_location] if col_location else "",
        "Tender Site":             "Tender Details",
        "Project Name":            df[col_brief]    if col_brief    else "",
        "Project Link":            "",
        "TDR No":                  df[col_tdr]      if col_tdr      else "",
        "Department Name":         df[col_auth]     if col_auth     else "",
        "Tender ID":               df[col_tid]      if col_tid      else "",
        "Refrence No":             df[col_tender_no]if col_tender_no else "",
        "Tender Value(Crore)":     crore_series.round(2),
        "Tender Status":           "",
        "Publish Date":            pub_series,
        "Bid Submission Date":     due_series,
        "Department Address":      "",
        "Contractor Name(L1)":     "",
        "Bidder Name":             "",
    }, columns=FINAL_COLS)

    # Stringify non-numeric columns, replace NaN with blanks
    for col in out.columns:
        if col != "Tender Value(Crore)":
            out[col] = out[col].astype(str).replace({"nan": ""})
    return out

# ========= STATE-WISE WRITER (with 'All Data' first) =========
def _sanitize_sheet_name(name: str) -> str:
    """Make a safe Excel sheet name (<=31 chars, no []:*?/\\)."""
    if pd.isna(name) or str(name).strip() == "":
        name = "Unknown"
    invalid = r'[]:*?/\\'
    for ch in invalid:
        name = name.replace(ch, "_")
    name = name.strip()
    return name[:31] if len(name) > 31 else name

def write_statewise_workbook(path: str, df: pd.DataFrame) -> None:
    """
    Write a workbook with:
      1) 'All Data' sheet FIRST (entire df)
      2) One sheet per State (''/NA -> 'Unknown')
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)

    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        # 1) ALL DATA sheet first (even if df is empty)
        if df.empty:
            all_df = pd.DataFrame(columns=FINAL_COLS)
        else:
            # Ensure the same column order
            all_df = df.reindex(columns=FINAL_COLS, fill_value="")
        all_df.to_excel(writer, sheet_name="All Data", index=False)

        # If empty or no 'State' column, we're done
        if df.empty or "State" not in df.columns:
            return

        # 2) State-wise sheets
        used: Dict[str, int] = {"All Data": 1}  # reserve the name to avoid collision
        states_series = df["State"].fillna("").astype(str)
        unique_states = sorted(set(states_series))
        # Keep Unknown first like before (optional, can remove if not desired)
        if "" in unique_states:
            unique_states.remove("")
            unique_states = [""] + unique_states  # '' → 'Unknown' first

        for state_value in unique_states:
            sheet_df = df[states_series == state_value].copy()
            safe = _sanitize_sheet_name(state_value)  # also converts "" to "Unknown"
            if safe in used:
                used[safe] += 1
                # If name already used, append suffix keeping 31-char limit
                base = safe[:28] if len(safe) > 28 else safe
                safe = f"{base}_{used[safe]:02d}"
            else:
                used[safe] = 1
            sheet_df.to_excel(writer, sheet_name=safe, index=False)

# ========= MAIN PIPELINE =========
if __name__ == "__main__":
    print("========== START ==========")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ---- LIVE: keywords ----
    print("[STEP] LIVE keywords merge")
    df_keyword = dedupe_by_tender_id(merge_folders_to_df(LIVE_DI, LIVE_DI_IRON, LIVE_WATER))
    print(f"[INFO] df_keyword rows: {len(df_keyword)}")

    # ---- LIVE: mail ----
    print("[STEP] LIVE mail merge")
    df_mail = dedupe_by_tender_id(merge_folders_to_df(LIVE_MAIL))
    print(f"[INFO] df_mail rows: {len(df_mail)}")

    # Remove overlaps (mail - keyword)
    ids_keyword = set(df_keyword["tender_id"].dropna().astype(str)) if not df_keyword.empty else set()
    df_mail_unique = filter_out_ids(df_mail, ids_keyword)
    print(f"[INFO] df_mail_unique rows: {len(df_mail_unique)}")

    # Final LIVE merge & dedupe
    df_merge_live = dedupe_by_tender_id(
        pd.concat([df_keyword, df_mail_unique], ignore_index=True, sort=False)
        if not (df_keyword.empty and df_mail_unique.empty) else pd.DataFrame(columns=["tender_id"])
    )
    print(f"[INFO] df_merge_live rows (pre-past-filter): {len(df_merge_live)}")

    # ---- PAST ----
    print("[STEP] PAST merge")
    df_merge_past = dedupe_by_tender_id(merge_folders_to_df(PAST_DI_PIPE, PAST_DI_IRON, PAST_WATER))
    print(f"[INFO] df_merge_past rows: {len(df_merge_past)}")

    # ---- FINAL FILTER: LIVE minus PAST by tender_id ----
    ids_past = set(df_merge_past["tender_id"].dropna().astype(str)) if not df_merge_past.empty else set()
    df_merge_live = dedupe_by_tender_id(filter_out_ids(df_merge_live, ids_past))
    print(f"[INFO] df_merge_live rows (final live before shaping): {len(df_merge_live)}")

    # ---- Add/refresh "Cr." column on BOTH frames ----
    df_merge_live = insert_crore_column(df_merge_live)
    df_merge_past = insert_crore_column(df_merge_past)

    # ---- SHAPE to final schema (dates normalized to DD-MM-YYYY) ----
    live_final = to_final_schema(df_merge_live)
    past_final = to_final_schema(df_merge_past)

    # ---- SAVE: 'All Data' first, then state-wise sheets ----
    print("[STEP] Writing state-wise workbooks (+ All Data)")
    write_statewise_workbook(OUT_LIVE_XLSX, live_final)
    write_statewise_workbook(OUT_PAST_XLSX, past_final)

    print(f"[OK] Live_Tender  -> {OUT_LIVE_XLSX}")
    print(f"[OK] Past_Tender  -> {OUT_PAST_XLSX}")
    print("=========== DONE ==========")


========== START ==========
[STEP] LIVE keywords merge
[INFO] df_keyword rows: 11198
[STEP] LIVE mail merge
[INFO] df_mail rows: 13244
[INFO] df_mail_unique rows: 13244
[INFO] df_merge_live rows (pre-past-filter): 24442
[STEP] PAST merge
[INFO] df_merge_past rows: 6637
[INFO] df_merge_live rows (final live before shaping): 18465
[STEP] Writing state-wise workbooks (+ All Data)
[OK] Live_Tender  -> D:\Dailydata_tenderdetails\Final_Data\Live_Tender.xlsx
[OK] Past_Tender  -> D:\Dailydata_tenderdetails\Final_Data\Past_Tender.xlsx
=========== DONE ==========
